# 04 - Feature Engineering

This notebook generates sliding windows from the cleaned sensor data for machine learning and deep learning training.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

INPUT_PATH = Path("..") / "data" / "interim" / "mhealth_cleaned.csv"
OUTPUT_DIR = Path("..") / "data" / "processed"

sensor_columns = [
    "alx", "aly", "alz",
    "glx", "gly", "glz",
    "arx", "ary", "arz",
    "grx", "gry", "grz",
]

df = pd.read_csv(INPUT_PATH)
print("Cleaned data shape:", df.shape)

## Build Sliding Window Dataset

In [ ]:
def create_windows(data: pd.DataFrame, window_size: int, step_size: int):
    X, y = [], []
    groups = data.groupby("subject", sort=False)
    for _, group in groups:
        features = group[sensor_columns].values
        labels = group["Activity"].values
        for start in range(0, len(group) - window_size + 1, step_size):
            window = features[start : start + window_size]
            label = np.bincount(labels[start : start + window_size]).argmax()
            X.append(window)
            y.append(label)
    return np.array(X), np.array(y)

WINDOW_SIZE = 128
STEP_SIZE = 64
X, y = create_windows(df, WINDOW_SIZE, STEP_SIZE)
print("Windowed X shape:", X.shape)
print("Windowed y shape:", y.shape)
print("Unique labels:", np.unique(y))

In [ ]:
from collections import Counter
print("Label counts:", Counter(y))

## Save Windowed Data

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_DIR / "X_windows.npy", X)
np.save(OUTPUT_DIR / "y_windows.npy", y)
print(f"Saved windowed arrays to {OUTPUT_DIR}")